In [ ]:
# ============================================================
# WEEK 7 – MENACE + Bandits + Non-stationary Bandits
# Google Colab 
# ============================================================

!pip install numpy matplotlib --quiet
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
%matplotlib inline

np.random.seed(42)
random.seed(42)

# ============================================================
# PART 1 — MENACE (Simplified Matchbox Engine for Tic-Tac-Toe)
# ============================================================

# -------- Board Utilities --------
WIN_LINES = [
    (0,1,2),(3,4,5),(6,7,8), # rows
    (0,3,6),(1,4,7),(2,5,8), # cols
    (0,4,8),(2,4,6)          # diagonals
]

def check_winner(board):
    for a,b,c in WIN_LINES:
        s = board[a] + board[b] + board[c]
        if s == 3: return 1      # MENACE wins
        if s == -3: return -1    # Opponent wins
    if 0 not in board: return 0   # Draw
    return None                   # Game ongoing

# -------- Canonical State (crucial to MENACE) --------
def rotations(board):
    b = np.array(board).reshape(3,3)
    rots = []
    for k in range(4):
        b = np.rot90(b)
        rots.append(tuple(b.flatten()))
    return rots

def reflections(board):
    b = np.array(board).reshape(3,3)
    return [
        tuple(np.fliplr(b).flatten()),
        tuple(np.flipud(b).flatten())
    ]

def canonical(board):
    """Return the lexicographically smallest symmetry of board."""
    boards = rotations(board) + reflections(board)
    return min(boards)

# -------- MENACE Matchboxes --------
matchboxes = defaultdict(Counter)

def MENACE_select_move(board):
    canon = canonical(board)

    # create box if unseen
    if canon not in matchboxes:
        legal = [i for i in range(9) if board[i] == 0]
        for pos in legal:
            matchboxes[canon][pos] = 4    # starting beads

    # weighted random choice
    moves, counts = zip(*matchboxes[canon].items())
    probs = np.array(counts) / sum(counts)
    move = np.random.choice(moves, p=probs)
    return move, canon

def opponent_random(board):
    legal = [i for i in range(9) if board[i] == 0]
    return random.choice(legal)

# -------- Train MENACE --------
def train_MENACE(episodes=5000):
    for _ in range(episodes):
        board = [0]*9
        history = []

        turn = 1   # MENACE = +1, Opponent = -1

        while True:
            if turn == 1:
                move, canon = MENACE_select_move(board)
                board[move] = 1
                history.append((canon, move))
            else:
                move = opponent_random(board)
                board[move] = -1

            winner = check_winner(board)
            if winner is not None:
                # reward or punishment
                for canon, mv in history:
                    if winner == 1:
                        matchboxes[canon][mv] += 3
                    elif winner == -1:
                        matchboxes[canon][mv] = max(1, matchboxes[canon][mv]-1)
                break

            turn *= -1

print("Training MENACE...")
train_MENACE()
print("MENACE training complete with", len(matchboxes), "matchboxes.")

# ============================================================
# PART 2 — Binary Bandit (Two-armed) using epsilon-greedy
# ============================================================

def binary_bandit(p1=0.6, p2=0.8, steps=5000, eps=0.1):
    Q = [0.0, 0.0]
    counts = [0, 0]
    rewards = []

    for t in range(steps):
        # ε-greedy choice
        if random.random() < eps:
            action = random.choice([0,1])
        else:
            action = np.argmax(Q)

        # stochastic reward
        reward = 1 if random.random() < (p1 if action==0 else p2) else 0
        rewards.append(reward)

        counts[action] += 1
        Q[action] += (1/counts[action]) * (reward - Q[action])

    print("Binary Bandit Finished. Empirical mean reward =", np.mean(rewards))
    return rewards

rewards_binary = binary_bandit()

plt.plot(np.convolve(rewards_binary, np.ones(200)/200, mode='valid'))
plt.title("Binary Bandit — Moving Average Reward")
plt.show()

# ============================================================
# PART 3 — 10-Armed Non-stationary Random-Walk Bandit
# ============================================================

class NonStationaryBandit:
    def __init__(self, k=10, sigma_walk=0.01):
        self.k = k
        self.sigma = sigma_walk
        self.means = np.zeros(k)

    def step(self, action):
        # random walk update
        self.means += np.random.normal(0, self.sigma, self.k)

        # reward sampled from current mean
        reward = np.random.normal(self.means[action], 1)
        return reward

bandit = NonStationaryBandit()

# ============================================================
# PART 4 — Agents: Sample Average vs Constant Step-size (α)
# ============================================================

def epsilon_greedy_sample_avg(bandit, eps=0.1, steps=10000):
    k = bandit.k
    Q = np.zeros(k)
    counts = np.zeros(k)
    rewards = []
    best_action_chosen = []

    for t in range(steps):
        a_star = np.argmax(bandit.means)
        if random.random() < eps:
            a = np.random.randint(k)
        else:
            a = np.argmax(Q)

        r = bandit.step(a)
        rewards.append(r)
        best_action_chosen.append(1 if a == a_star else 0)

        counts[a] += 1
        Q[a] += (1/counts[a]) * (r - Q[a])   # sample average

    return np.array(rewards), np.array(best_action_chosen)

def epsilon_greedy_constant_alpha(bandit, eps=0.1, alpha=0.1, steps=10000):
    k = bandit.k
    Q = np.zeros(k)
    rewards = []
    best_action_chosen = []

    for t in range(steps):
        a_star = np.argmax(bandit.means)
        if random.random() < eps:
            a = np.random.randint(k)
        else:
            a = np.argmax(Q)

        r = bandit.step(a)
        rewards.append(r)
        best_action_chosen.append(1 if a == a_star else 0)

        Q[a] += alpha * (r - Q[a])  # recency-weighted update

    return np.array(rewards), np.array(best_action_chosen)

# Run both agents
bandit = NonStationaryBandit()
r1, b1 = epsilon_greedy_sample_avg(bandit)

bandit = NonStationaryBandit()
r2, b2 = epsilon_greedy_constant_alpha(bandit)

# ============================================================
# Plotting the comparison
# ============================================================

plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(np.convolve(r1, np.ones(200)/200, mode='same'), label="Sample Avg")
plt.plot(np.convolve(r2, np.ones(200)/200, mode='same'), label="Constant step-size α=0.1")
plt.title("10-Armed Non-stationary Bandit — Avg Reward")
plt.legend(loc="upper left")

plt.subplot(1,2,2)
plt.plot(np.convolve(b1, np.ones(200)/200, mode='same'), label="% Optimal — Sample Avg")
plt.plot(np.convolve(b2, np.ones(200)/200, mode='same'), label="% Optimal — Constant α=0.1")
plt.title("Optimal Action Percentage")
plt.legend(loc="lower right")
plt.show()


